In [1]:
import numpy as np 
from sklearn.cluster import KMeans 
from sklearn.feature_extraction.text import TfidfVectorizer 
from tabulate import tabulate 
from collections import Counter 

In [2]:
dataset = ["I love playing football on the weekends", 
           "I enjoy hiking and camping in the mountains", 
           "I like to read books and watch movies", 
           "I prefer playing video games over sports", 
           "I love listening to music and going to concerts"] 

In [3]:
vectorizer = TfidfVectorizer() 
X = vectorizer.fit_transform(dataset)

In [5]:
from sklearn.cluster import KMeans
from tabulate import tabulate

# 1. Define the number of clusters 
k = 2  
km = KMeans(n_clusters=k, random_state=42) # Added random_state for consistent results
km.fit(X) 
 
# 2. Predict the clusters for each document 
y_pred = km.predict(X) 
 
# 3. Display the document and its predicted cluster in a table 
table_data = [["Document", "Predicted Cluster"]] 
table_data.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred)]) 
print(tabulate(table_data, headers="firstrow")) 

# 4. Print top terms per cluster 
print("\nTop terms per cluster:") 
order_centroids = km.cluster_centers_.argsort()[:, ::-1] 
terms = vectorizer.get_feature_names_out() 

for i in range(k): 
    print(f"Cluster {i}:") # Using f-strings for cleaner formatting
    for ind in order_centroids[i, :10]: 
        print(f" {terms[ind]}") 
    print() # Adds a newline between clusters

Document                                           Predicted Cluster
-----------------------------------------------  -------------------
I love playing football on the weekends                            0
I enjoy hiking and camping in the mountains                        0
I like to read books and watch movies                              1
I prefer playing video games over sports                           0
I love listening to music and going to concerts                    1

Top terms per cluster:
Cluster 0:
 playing
 the
 weekends
 on
 football
 prefer
 video
 over
 sports
 games

Cluster 1:
 to
 and
 read
 like
 books
 movies
 watch
 music
 listening
 going



/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


In [6]:
# Calculate purity 
total_samples = len(y_pred) 
cluster_label_counts = [Counter(y_pred)] 
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples 
print("Purity:", purity) 

Purity: 0.6


In [8]:
pip install gensim

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 7.9 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gensim]━━━━━ 1/2 [gensim]
Note: you may need to restart the kernel to use updated packages.


In [9]:
conda install -c conda-forge gensim

/opt/conda/lib/python3.9/site-packages/conda/base/context.py:211: FutureWarning: Adding 'defaults' to channel list implicitly is deprecated and will be removed in 25.9. 

To remove this warning, please choose a default channel explicitly with conda's regular configuration system, e.g. by adding 'defaults' to the list of channels:

  conda config --add channels defaults

For more information see https://docs.conda.io/projects/conda/en/stable/user-guide/configuration/use-condarc.html

  deprecated.topic(
Retrieving notices: done

EnvironmentNotWritableError: The current user does not have write permissions to the target environment.
  environment location: /opt/conda/envs/anaconda-2025.12-py312
  uid: 2668868
  gid: 60000



Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install gensim scikit-learn tabulate

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [12]:
dataset = ["I love playing football on the weekends", 
           "I enjoy hiking and camping in the mountains", 
           "I like to read books and watch movies", 
           "I prefer playing video games over sports", 
           "I love listening to music and going to concerts"] 

In [14]:
from gensim.models import Word2Vec

# Now your code will work:
tokenized_dataset = [doc.split() for doc in dataset] 
word2vec_model = Word2Vec(sentences=tokenized_dataset, vector_size=100, 
                          window=5, min_count=1, workers=4)

In [15]:
X = np.array([np.mean([word2vec_model.wv[word] for word in doc.split() if word in 
word2vec_model.wv], axis=0) for doc in dataset])

In [16]:
k = 2  # Define the number of clusters 
km = KMeans(n_clusters=k) 
km.fit(X) 
 
# Predict the clusters for each document 
y_pred = km.predict(X) 
 
# Tabulate the document and predicted cluster 
table_data = [["Document", "Predicted Cluster"]] 
table_data.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred)]) 
print(tabulate(table_data, headers="firstrow")) 
# Calculate purity 
total_samples = len(y_pred) 
cluster_label_counts = [Counter(y_pred)] 
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples 
print("Purity:", purity) 

Document                                           Predicted Cluster
-----------------------------------------------  -------------------
I love playing football on the weekends                            0
I enjoy hiking and camping in the mountains                        0
I like to read books and watch movies                              1
I prefer playing video games over sports                           0
I love listening to music and going to concerts                    1
Purity: 0.6


In [19]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score
from tabulate import tabulate

# 1. SETUP & DATA LOADING
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

# Load the dataset
try:
    df = pd.read_csv('customer_complaints_1.csv')
    # Use 'text' (lowercase) as seen in your previous error log
    target_col = 'text' 
    dataset = df[target_col].astype(str).tolist()
    print(f"Successfully loaded {len(dataset)} documents from '{target_col}'.")
except KeyError:
    print(f"Error: Could not find column '{target_col}'. Check CSV headers.")
    # Fallback to identify what is available
    print(f"Available columns: {df.columns.tolist()}")
    exit()

# 2. PREPROCESSING FUNCTION
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text) # Remove punctuation
    text = re.sub(r'\d+', '', text)      # Remove numbers
    tokens = text.split()
    # Remove stopwords
    return [word for word in tokens if word not in stop_words]

# Prepare data for both models
tokenized_docs = [preprocess_text(doc) for doc in dataset]
clean_docs = [" ".join(doc) for doc in tokenized_docs]

# 3. VECTORIZATION - TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=1000)
X_tfidf = tfidf_vectorizer.fit_transform(clean_docs)

# 4. VECTORIZATION - WORD2VEC (Document Embeddings)
w2v_model = Word2Vec(sentences=tokenized_docs, vector_size=100, window=5, min_count=1, workers=4)

def get_document_vector(tokens, model):
    # Filter words that exist in the Word2Vec vocabulary
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0) # Average word vectors

X_w2v = np.array([get_document_vector(doc, w2v_model) for doc in tokenized_docs])

# 5. K-MEANS CLUSTERING
k = 3 # Number of clusters
km_tfidf = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
km_w2v = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)

y_tfidf = km_tfidf.fit_predict(X_tfidf)
y_w2v = km_w2v.fit_predict(X_w2v)

# 6. PURITY CALCULATION FUNCTION
def calculate_purity(y_true, y_pred):
    # Purity = (1/N) * sum(max count of ground truth category in each cluster)
    contingency_matrix = pd.crosstab(y_true, y_pred)
    return np.sum(contingency_matrix.max(axis=0)) / np.sum(contingency_matrix.values)

# Assuming 'rating' or 'author' is used as a proxy for ground truth classes
# (Note: In a real scenario, you'd use actual category labels)
if 'rating' in df.columns:
    purity_tfidf = calculate_purity(df['rating'], y_tfidf)
    purity_w2v = calculate_purity(df['rating'], y_w2v)
    print(f"\nCluster Purity (based on Rating):")
    print(f"TF-IDF Purity: {purity_tfidf:.4f}")
    print(f"Word2Vec Purity: {purity_w2v:.4f}")

# 7. DISPLAY TOP TERMS PER CLUSTER (TF-IDF)
print("\n--- TF-IDF TOP TERMS PER CLUSTER ---")
order_centroids = km_tfidf.cluster_centers_.argsort()[:, ::-1]
terms = tfidf_vectorizer.get_feature_names_out()

cluster_summary = []
for i in range(k):
    top_terms = [terms[ind] for ind in order_centroids[i, :10]]
    cluster_summary.append([f"Cluster {i}", ", ".join(top_terms)])

print(tabulate(cluster_summary, headers=["Cluster", "Top Keywords"], tablefmt="grid"))

Successfully loaded 19 documents from 'text'.

Cluster Purity (based on Rating):
TF-IDF Purity: 0.9474
Word2Vec Purity: 0.9474

--- TF-IDF TOP TERMS PER CLUSTER ---
+-----------+-----------------------------------------------------------------------------------------------+
| Cluster   | Top Keywords                                                                                  |
+===========+===============================================================================================+
| Cluster 0 | boxes, second, adding, protocol, malfunction, investigating, since, floor, customer, possible |
+-----------+-----------------------------------------------------------------------------------------------+
| Cluster 1 | internet, contract, comcast, would, service, speed, xfinity, customer, get, tech              |
+-----------+-----------------------------------------------------------------------------------------------+
| Cluster 2 | rude, service, mbps, bill, rep, day, people, joke, 

In [21]:
#Yes, significantly. Purity is a measure of how "clean" your clusters are relative to ground-truth labels. Preprocessing impacts Purity in the following ways Noise Reduction,Feature Sparsity, and Word2Vec Specifics.

k = 3 # Adjust based on expected complaint types (e.g., Billing, Tech Support, Delivery)
km_tfidf = KMeans(n_clusters=k, random_state=42).fit(X_tfidf)
km_w2v = KMeans(n_clusters=k, random_state=42).fit(X_w2v)

# Add predictions back to dataframe for analysis
df['Cluster_TFIDF'] = km_tfidf.labels_
df['Cluster_W2V'] = km_w2v.labels_

print("TF-IDF Top terms per cluster:")
order_centroids = km_tfidf.cluster_centers_.argsort()[:, ::-1]
terms = tfidf_vectorizer.get_feature_names_out()
for i in range(k):
    top_10 = [terms[ind] for ind in order_centroids[i, :10]]
    print(f"Cluster {i}: {', '.join(top_10)}")

TF-IDF Top terms per cluster:
Cluster 0: service, customer, since, adding, boxes, second, rude, speed, joke, never
Cluster 1: internet, comcast, mbps, service, would, xfinity, day, contract, call, told
Cluster 2: contract, internet, years, comcast, im, issue, however, means, advertised, verizon
